In [6]:
!pip install zhipuai
!pip install --upgrade zhipuai
!pip install unbabel-comet pandas openpyxl huggingface_hub requests
import json
import re
from google.colab import files
from zhipuai import ZhipuAI
import os
import csv
import logging
import xml.etree.ElementTree as ET
from scipy.spatial.distance import cosine
import pandas as pd
import requests
import string
from hashlib import md5
import random
import time
import numpy as np
from collections import defaultdict
from comet import download_model, load_from_checkpoint
client = ZhipuAI(api_key="9bb1a129d84f1f5db97393b35b909123.Vx6nhnD0E0SsUpBk")

In [7]:
def extract_terms(translation_memory_json):
  prompt = f'''
  双语对齐文本如下所示：
  {translation_memory_json}
  请你提取以上文本中所有中文和英文术语，并输出这些术语的JSON格式，参考样式为："chinese":"","english":""
  '''
  response = client.chat.completions.create(
    model="glm-4",  # 填写需要调用的模型名称
    messages=[{"role": "user", "content": prompt}],)
  return response.choices[0].message.content

def capture_and_fix_json(json_text):
    # 正则表达式匹配完整的 JSON 对象
    pattern = re.compile(r'\{[^}]*\}')

    # 查找所有完整的 JSON 对象
    matches = pattern.findall(json_text)

    # 构建新的 JSON 数组字符串
    valid_json_text = '[' + ','.join(matches) + ']'

    # 尝试解析新的 JSON 数组
    try:
        data = json.loads(valid_json_text)
        return data
    except json.JSONDecodeError as e:
        # 如果解析失败，返回错误信息
        print(f"Error decoding JSON: {e}")
        return []

def cosine_similarity(vec1, vec2):
    """计算余弦相似度"""
    return 1 - cosine(vec1, vec2)

def get_embedding(client, model, input_text):
    """获取向量表示"""
    response = client.embeddings.create(model=model, input=input_text)
    embedding_data = response.data
    embedding = embedding_data[0].embedding
    return embedding

def create_csv_file(TMX_FILE_PATH, csv_file_path):
    """创建并写入CSV文件"""

    try:
        tree = ET.parse(TMX_FILE_PATH)
        root = tree.getroot()
    except Exception as e:
        logging.error(f"解析TMX文件时发生错误: {e}")
        return

    with open(csv_file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['zh', 'en', 'zh_vector', 'en_vector'])

        for tu in root.findall('.//tu'):
            zh_text, en_text = None, None
            for tuv in tu.findall('tuv'):
                lang, seg = tuv.get('{http://www.w3.org/XML/1998/namespace}lang'), tuv.find('seg').text
                if lang == 'zh-CN':
                    zh_text = seg
                elif lang == 'en-US':
                    en_text = seg
            if zh_text and en_text:
                zh_vector = get_embedding(client, model_name, zh_text)
                en_vector = get_embedding(client, model_name, en_text)
                writer.writerow([zh_text, en_text, ','.join(map(str, zh_vector)), ','.join(map(str, en_vector))])

def process_excel_to_csv(excel_file_path, csv_file_path):
    # 读取Excel文件
    df = pd.read_excel(excel_file_path)

    # 去除列名中的额外空格
    df.columns = df.columns.str.strip()
    print(df.columns)
    print(df.head())
    with open(csv_file_path, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['zh', 'en', 'zh_vector', 'en_vector'])

        for index, row in df.iterrows():
            zh_text = row['chinese']
            en_text = row['english']

            if pd.notna(zh_text) and pd.notna(en_text):
                zh_vector = get_embedding(client, model_name, zh_text)
                en_vector = get_embedding(client, model_name, en_text)
                writer.writerow([zh_text, en_text, ','.join(map(str, zh_vector)), ','.join(map(str, en_vector))])

def load_translation_memory_from_csv(csv_file_path):
    """从CSV文件中加载翻译记忆库和向量"""
    translation_memory = []
    tm_vectors = {"zh": [], "en": []}
    with open(csv_file_path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # 跳过标题行
        for row in reader:
            zh_text, en_text, zh_vector_str, en_vector_str = row
            zh_vector = [float(x) for x in zh_vector_str.split(',')]
            en_vector = [float(x) for x in en_vector_str.split(',')]
            translation_memory.append({"zh": zh_text, "en": en_text})
            tm_vectors["zh"].append(zh_vector)
            tm_vectors["en"].append(en_vector)
    return translation_memory, tm_vectors

def find_top_translation_memory(query_vector, tm_vectors, query_language):
    """找到翻译记忆库中与目标向量相似度最高的翻译记忆"""
    similarities = []
    for idx, vec in enumerate(tm_vectors[query_language]):
        similarity = cosine_similarity(query_vector, vec)
        similarities.append((similarity, idx))
    top_similarity = max(similarities, key=lambda x: x[0])
    return top_similarity[1], top_similarity[0]

In [8]:
all_translation_memories = []
all_files = []
seen_pairs = set()

def get_tmx_files():
    global all_files

    while True:
        action = input("请选择操作：1 - 上传TMX文件，2 - 检查目录中的TMX文件，stop - 停止操作: ").strip().lower()
        if action == 'stop':
            break
        elif action == '1':
            print("请上传TMX文件。")
            uploaded_files = files.upload()
            for file_name, file_content in uploaded_files.items():
                full_path = os.path.abspath(file_name)
                if full_path.endswith('.tmx'):
                    all_files.append(full_path)  # 将文件路径添加到 all_files 列表中
                    if full_path not in all_files:
                        all_files.append(full_path)
                        print("文件已上传:", full_path)
                    else:
                        print(f"文件 {file_name} 已存在，跳过上传。")
                else:
                    print("不支持的文件类型:", file_name)
        elif action == '2':
            directory="/content/"
            #directory = input("请输入要检查的目录路径: ")
            print("正在检查目录中的TMX文件...")
            try:
                for file_name in os.listdir(directory):
                    if file_name.endswith('.tmx'):
                        full_path = os.path.join(directory, file_name)
                        if full_path not in all_files:
                            all_files.append(full_path)
                            print("找到并添加TMX文件:", full_path)
                        else:
                            print(f"文件 {file_name} 已存在，跳过添加。")
            except Exception as e:
                print("读取目录错误:", e)

    print("操作完成。")

def print_translation_memories():
    for translation_memory in all_translation_memories:
        translation_memory_json = json.dumps(translation_memory, ensure_ascii=False, indent=4)
        file_name = translation_memory['file_name']
        print(f"Translation Memory for file {file_name}: {translation_memory_json}")
    return translation_memory_json

def process_tmx_files():
    global all_files, all_translation_memories, seen_pairs

    for tmx_file in all_files:
        try:
            translation_memory = []
            tree = ET.parse(tmx_file)
            root = tree.getroot()

            for tu in root.iter('tu'):
                zh = ""
                en = ""

                for tuv in tu.findall('tuv'):
                    lang = tuv.get('{http://www.w3.org/XML/1998/namespace}lang')
                    seg = tuv.find('seg').text

                    if lang == "zh-CN":
                        zh = seg
                    elif lang == "en-US":
                        en = seg

                if zh and en:
                    pair = (zh, en)
                    if pair not in seen_pairs:
                        translation_memory.append({"zh": zh, "en": en})
                        seen_pairs.add(pair)

            if translation_memory:
                all_translation_memories.append(translation_memory)
        except Exception as e:
            print(f"处理文件 {tmx_file} 时出错:", e)

def print_translation_memories():
    for idx, translation_memory in enumerate(all_translation_memories):
        translation_memory_json = json.dumps(translation_memory, ensure_ascii=False, indent=4)
        file_name = os.path.basename(all_files[idx])  # 获取文件的基本名称
        print(f"Translation Memory for file {file_name}: {translation_memory_json}")
    return translation_memory_json

get_tmx_files()
process_tmx_files()
print("已处理的TMX文件列表如下:")
for file_path in all_files:
    print(file_path)
translation_memory_json = print_translation_memories()

请选择操作：1 - 上传TMX文件，2 - 检查目录中的TMX文件，stop - 停止操作: 2
正在检查目录中的TMX文件...
找到并添加TMX文件: /content/1.22_zh-CN_en-US107行.tmx
找到并添加TMX文件: /content/4.11_zh-CN_en-US119行.tmx
请选择操作：1 - 上传TMX文件，2 - 检查目录中的TMX文件，stop - 停止操作: stop
操作完成。
已处理的TMX文件列表如下:
/content/1.22_zh-CN_en-US107行.tmx
/content/4.11_zh-CN_en-US119行.tmx
Translation Memory for file 1.22_zh-CN_en-US107行.tmx: [
    {
        "zh": "2024年1月22日外交部发言人汪文斌主持例行记者会",
        "en": "Foreign Ministry Spokesperson Wang Wenbin’s Regular Press Conference on January 22, 2024"
    },
    {
        "zh": "深圳卫视记者：今年是中巴建交50周年。",
        "en": "Shenzhen TV: This year marks the 50th anniversary of diplomatic relations between China and Brazil."
    },
    {
        "zh": "1月19日，中共中央政治局委员、外交部长王毅访问巴西。",
        "en": "On January 19, Member of the Political Bureau of the CPC Central Committee and Foreign Minister Wang Yi visited Brazil."
    },
    {
        "zh": "请问发言人能否具体介绍王毅外长此次巴西之行的亮点，以及双方在会晤中取得的成果和达成的共识？",
        "en": "Can you share more about the highlights

In [9]:
def get_terms_in_xlsx(all_files, all_translation_memories):
    for idx, translation_memory in enumerate(all_translation_memories):
        # 生成 Excel 文件的名称和路径
        excel_file_name = os.path.splitext(os.path.basename(all_files[idx]))[0] + ".xlsx"
        excel_file_path = os.path.join(os.path.dirname(all_files[idx]), excel_file_name)

        # 检查 Excel 文件是否已存在
        if not os.path.exists(excel_file_path):
            # 提取术语并修正数据
            json_data = extract_terms(translation_memory)
            print(f"术语提取结果 for file {all_files[idx]}: {json_data}")

            data = capture_and_fix_json(json_data)

            # 尝试构建 DataFrame
            try:
                df = pd.DataFrame(data)
                print("DataFrame 构建成功:\n", df)
            except ValueError as e:
                print("DataFrame 构造函数未正确调用:", e)
                continue  # 如果 DataFrame 构造失败，跳过当前循环

            # 创建 Excel 文件
            df.to_excel(excel_file_path, index=False)
            print("Excel 文件已成功生成:", excel_file_path)
        else:
            print(f"Excel 文件 {excel_file_path} 已存在，跳过生成。")

get_terms_in_xlsx(all_files, all_translation_memories)

Excel 文件 /content/1.22_zh-CN_en-US107行.xlsx 已存在，跳过生成。
Excel 文件 /content/4.11_zh-CN_en-US119行.xlsx 已存在，跳过生成。


In [11]:
all_found_xlsx_files = []
def get_xlsx_files():
    while True:
        action = input("请选择操作：1 - 上传XLSX文件，2 - 检查目录中的XLSX文件，stop - 停止操作: ").strip().lower()
        if action == 'stop':
            break
        elif action == '1':
            print("请上传XLSX文件。")
            uploaded_files = files.upload()  # 这需要具体的上传文件逻辑
            for file_name in uploaded_files.keys():
                full_path = os.path.abspath(file_name)
                if full_path.endswith('.xlsx'):
                    # 使用集合检查文件名是否重复
                    if full_path not in all_found_xlsx_files:
                        all_found_xlsx_files.append(full_path)
                        print("文件已上传:", full_path)
                    else:
                        print(f"文件 {file_name} 已存在，跳过上传。")
                else:
                    print("不支持的文件类型:", file_name)
        elif action == '2':
            #directory = input("请输入要检查的目录路径: ")
            directory = '/content/'
            print("正在检查目录中的XLSX文件...")
            try:
                for file_name in os.listdir(directory):
                    if file_name.endswith('.xlsx'):
                        full_path = os.path.join(directory, file_name)
                        if full_path not in all_found_xlsx_files:
                            all_found_xlsx_files.append(full_path)
                            print("找到并添加XLSX文件:", full_path)
                        else:
                            print(f"文件 {file_name} 在系统中已存在，跳过添加。")
            except Exception as e:
                print("读取目录错误:", e)
        else:
            print("无效的输入，请输入1, 2, 或 'stop'.")

get_xlsx_files()
print("操作完成。已处理的文件列表如下:")
for file_path in all_found_xlsx_files:
  print(file_path)

请选择操作：1 - 上传XLSX文件，2 - 检查目录中的XLSX文件，stop - 停止操作: 2
正在检查目录中的XLSX文件...
找到并添加XLSX文件: /content/4.11_zh-CN_en-US119行.xlsx
找到并添加XLSX文件: /content/1.22_zh-CN_en-US107行.xlsx
请选择操作：1 - 上传XLSX文件，2 - 检查目录中的XLSX文件，stop - 停止操作: stop
操作完成。已处理的文件列表如下:
/content/4.11_zh-CN_en-US119行.xlsx
/content/1.22_zh-CN_en-US107行.xlsx


In [12]:
input_sentence = input("请输入要查询的句子：")
query_language = input("请输入句子语言（中文请输入'zh'，英文请输入'en'）：")
input_vector = get_embedding(client, "embedding-2", input_sentence)

请输入要查询的句子：同巴西外长维埃拉举行了第四次中巴外长级全面战略对话并共同会见记者。
请输入句子语言（中文请输入'zh'，英文请输入'en'）：zh


In [13]:
best_similarity = -1
best_translation_memory = None
all_top_translations = []

for tmx_file in all_files:
    csv_file_path = os.path.splitext(tmx_file)[0] + "_embedding.csv"

    # 检查目标 CSV 文件是否存在
    if not os.path.exists(csv_file_path):
        model_name = "embedding-2"
        create_csv_file(tmx_file, csv_file_path)  # Corrected argument
    else:
        print(f"翻译记忆库{tmx_file}的CSV文件已存在，跳过写入操作。")

    # 加载翻译记忆库和向量
    translation_memory, tm_vectors = load_translation_memory_from_csv(csv_file_path)

    # 查找翻译记忆库中与输入向量相似度最高的翻译记忆的索引和相似度
    top_index, similarity = find_top_translation_memory(input_vector, tm_vectors, query_language)

    # 获取相似度最高的翻译记忆
    top_tm = translation_memory[top_index]

    # 保存每个文件的最相似翻译记忆和相似度
    all_top_translations.append((tmx_file, top_tm, similarity))

    # 比较相似度，记录最相似的翻译记忆
    if similarity > best_similarity:
        best_similarity = similarity
        best_translation_memory = top_tm

# 打印所有文件中找到的最相似翻译记忆
for tmx_file, top_tm, similarity in all_top_translations:
    print(f"{tmx_file}: 最相似的翻译记忆: {top_tm}，相似度: {similarity}")

# 打印相似度最高的翻译记忆
if best_translation_memory:
    print("\n总体上最相似的翻译记忆：", best_translation_memory)
    print("相似度：", best_similarity)

翻译记忆库/content/1.22_zh-CN_en-US107行.tmx的CSV文件已存在，跳过写入操作。
翻译记忆库/content/4.11_zh-CN_en-US119行.tmx的CSV文件已存在，跳过写入操作。
/content/1.22_zh-CN_en-US107行.tmx: 最相似的翻译记忆: {'zh': '同巴西外长维埃拉举行了第四次中巴外长级全面战略对话并共同会见记者。', 'en': 'He also co-chaired the fourth China-Brazil Foreign Ministerial-Level Comprehensive Strategic Dialogue with Foreign Minister Mauro Vieira and jointly met the press.'}，相似度: 0.9999999304065009
/content/4.11_zh-CN_en-US119行.tmx: 最相似的翻译记忆: {'zh': '去年，阿尔巴尼斯总理成功访华，两国领导人就进一步改善发展中澳关系达成重要共识。', 'en': 'Last year, Australian Prime Minister Albanese visited China. The leaders of the two countries reached important common understandings on further improving China-Australia relations.'}，相似度: 0.6326510375712839

总体上最相似的翻译记忆： {'zh': '同巴西外长维埃拉举行了第四次中巴外长级全面战略对话并共同会见记者。', 'en': 'He also co-chaired the fourth China-Brazil Foreign Ministerial-Level Comprehensive Strategic Dialogue with Foreign Minister Mauro Vieira and jointly met the press.'}
相似度： 0.9999999304065009


In [14]:
API_KEY = "sHRDZlob3VRRzXb4oGw7z5rd"
SECRET_KEY = "GFwTzTxMVbCN3uSAsTgt7ghuwqcDoURW"

def main():

    url = "https://aip.baidubce.com/rpc/2.0/nlp/v1/lexer?charset=UTF-8&access_token=" + get_access_token()

    payload = json.dumps({
        "text": f"{input_sentence}"
    })
    headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    print(response.text)
    data = json.loads(response.text)

# 创建一个新列表来存储词汇
    words = []

# 遍历 items 列表
    for item in data['items']:
    # 将每个元素的 'item' 键的值添加到词汇列表中
      words.append(item['item'])

    return words

def get_access_token():
    """
    使用 AK，SK 生成鉴权签名（Access Token）
    :return: access_token，或是None(如果错误)
    """
    url = "https://aip.baidubce.com/oauth/2.0/token"
    params = {"grant_type": "client_credentials", "client_id": API_KEY, "client_secret": SECRET_KEY}
    return str(requests.post(url, params=params).json().get("access_token"))

words=main()
chinese_punctuation = "。？！，、；：“”‘’（）《》【】"
all_punctuation = string.punctuation + chinese_punctuation

# 定义需要过滤的无意义词汇列表
english_stopwords = [
    'a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being',
    'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during',
    'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', 'her', 'here', 'hers', 'herself', 'him',
    'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'isn', "isn't", 'it', "it's", 'its', 'itself', 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most',
    'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out',
    'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she', "she's", 'should', "should've", 'shouldn', "shouldn't", 'so', 'some', 'such', 't', 'than', 'that', "that'll",
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'wasn',
    "wasn't", 'we', 'were', 'weren', "weren't", 'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'won', "won't", 'wouldn', "wouldn't", 'y', 'you',
    "you'd", "you'll", "you're", "you've", 'your', 'yours', 'yourself', 'yourselves'
]

chinese_stopwords = [
    '的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好', '自己', '这',
    '那', '还', '得', '与', '着', '下', '过', '可以', '到', '由', '这里', '一个', '再', '然后', '今', '其', '已', '无', '对', '从', '只有', '又', '或', '但', '后', '而', '之',
    '虽然', '这些', '例如', '如果', '这样', '由于', '并', '同时', '我们', '可以', '他们', '你们', '大家', '什么', '为什么', '哪里', '怎么', '多少', '如何', '那里', '时候',
    '这儿', '为了', '总是', '这个', '那个', '每个', '任何', '所有', '尽管', '尽管如此', '两者', '之间', '写出', '无论如何', '以及', '以', '天', '近', '各种', '表达', '转发', '百位'
]

filter_words = set(chinese_stopwords + english_stopwords)

# 假设 'words' 是已经提供的词列表
all_punctuation = string.punctuation + "，。！？、；：“”‘’（）《》【】"

# 清洗词汇并去除无意义的词汇
unique_words = set(word.strip(all_punctuation + ' ') for word in words if word.strip(all_punctuation + ' ') and word not in filter_words)

# 将集合转换回列表，如果需要保持原始顺序，可以使用下面的方式
cleaned_words = list(unique_words)
print(cleaned_words)
input_termvectors = []

for word in cleaned_words:
  input_termvector = get_embedding(client, "embedding-2", word)
  input_termvectors.append(input_termvector)

{"text":"同巴西外长维埃拉举行了第四次中巴外长级全面战略对话并共同会见记者。","items":[{"uri":"","formal":"","ne":"","item":"同","loc_details":[],"basic_words":["同"],"byte_offset":0,"byte_length":2,"pos":"p"},{"uri":"","formal":"","ne":"LOC","item":"巴西","loc_details":[],"basic_words":["巴西"],"byte_offset":2,"byte_length":4,"pos":""},{"uri":"","formal":"","ne":"","item":"外长","loc_details":[],"basic_words":["外","长"],"byte_offset":6,"byte_length":4,"pos":"n"},{"uri":"","formal":"","ne":"PER","item":"维埃拉","loc_details":[],"basic_words":["维埃拉"],"byte_offset":10,"byte_length":6,"pos":""},{"uri":"","formal":"","ne":"","item":"举行","loc_details":[],"basic_words":["举行"],"byte_offset":16,"byte_length":4,"pos":"v"},{"uri":"","formal":"","ne":"","item":"了","loc_details":[],"basic_words":["了"],"byte_offset":20,"byte_length":2,"pos":"u"},{"uri":"","formal":"","ne":"","item":"第四次","loc_details":[],"basic_words":["第","四","次"],"byte_offset":22,"byte_length":6,"pos":""},{"uri":"","formal":"","ne":"","item":"中巴","loc_details":[],"basic_word

In [15]:
# 如果目标CSV文件不存在，则将上传的文件处理为CSV文件
for xlsx_file in all_found_xlsx_files:
    xlsx_csv_file_path = os.path.splitext(xlsx_file)[0] + "_termsembedding.csv"
    # 检查目标CSV文件是否存在
    if not os.path.exists(xlsx_csv_file_path):
        process_excel_to_csv(xlsx_file, xlsx_csv_file_path)  # 更正后的参数
    else:
        print(f"术语表{xlsx_file}的CSV文件已存在，跳过写入操作：{xlsx_csv_file_path}")

# 遍历 words 列表中的每个词汇及其向量

def cosine_similarity(vector1, vector2):
    # 计算两个向量的点积
    dot_product = np.dot(vector1, vector2)
    # 计算两个向量的范数
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)
    # 计算余弦相似度
    if norm1 == 0 or norm2 == 0:
        return 0  # 避免除以零
    else:
        return dot_product / (norm1 * norm2)

def find_top_terms(query_vector, tm_vectors, query_language, threshold):
    similarities = []
    query_vector = np.array(query_vector)  # 确保是 NumPy 数组

    for idx, vec in enumerate(tm_vectors[query_language]):
        vec = np.array(vec)  # 确保是 NumPy 数组

        # 检查是否存在 NaN 值
        if np.any(np.isnan(query_vector)) or np.any(np.isnan(vec)):
            continue  # 跳过含有 NaN 的向量

        # 计算余弦相似度
        similarity = cosine_similarity(query_vector, vec)

        if similarity > threshold:
            similarities.append((similarity, idx))

    if similarities:
        similarities.sort(key=lambda x: x[0], reverse=True)
        return similarities
    else:
        return []

final_top_terms = []
threshold = 0.6


for word, input_termvector in zip(cleaned_words, input_termvectors):
    top_terms_by_file = defaultdict(list)
    for xlsx_file in all_found_xlsx_files:
        xlsx_csv_file_path = os.path.splitext(xlsx_file)[0] + "_termsembedding.csv"
        terms, terms_vectors = load_translation_memory_from_csv(xlsx_csv_file_path)
        term_similarities = find_top_terms(input_termvector, terms_vectors, query_language, threshold)

        for similarity, top_termindex in term_similarities:
            top_term = terms[top_termindex]
            top_terms_by_file[xlsx_file].append((top_term, similarity))

    # 打印或处理每个词汇的所有相似词汇
    if top_terms_by_file:
        print(f"所有术语表中，词汇 '{word}' 的相似词汇：")
        for term_file, top_terms in top_terms_by_file.items():
            print(f"在术语表 '{term_file}' 中，相似词汇是：")
            for top_term, similarity in top_terms:
                print(f"{top_term}，相似度：{similarity}")
            final_top_terms.extend([(top_term, similarity, term_file) for top_term, similarity in top_terms])
    else:
        print(f"所有术语表中，词汇 '{word}' 都没有找到相似词汇。")

# 选取final_top_terms中的文本术语部分
seen_terms = set()
terms = []

for term, _, _ in final_top_terms:
    term_tuple = tuple(term.items())  # 将字典转换为元组
    if term_tuple not in seen_terms:
        terms.append(term)
        seen_terms.add(term_tuple)

# 将术语列表转换为JSON格式
terms_json = json.dumps(terms, ensure_ascii=False, indent=4)

# 打印JSON格式输出
print(terms_json)

术语表/content/4.11_zh-CN_en-US119行.xlsx的CSV文件已存在，跳过写入操作：/content/4.11_zh-CN_en-US119行_termsembedding.csv
术语表/content/1.22_zh-CN_en-US107行.xlsx的CSV文件已存在，跳过写入操作：/content/1.22_zh-CN_en-US107行_termsembedding.csv
所有术语表中，词汇 '记者' 的相似词汇：
在术语表 '/content/4.11_zh-CN_en-US119行.xlsx' 中，相似词汇是：
{'zh': '无国界记者', 'en': 'Reporters Without Borders'}，相似度：0.6990251690403878
{'zh': '新华社', 'en': 'Xinhua News Agency'}，相似度：0.6885438744976872
{'zh': '例行记者会', 'en': 'Regular Press Conference'}，相似度：0.6372696392471016
所有术语表中，词汇 '同' 都没有找到相似词汇。
所有术语表中，词汇 '共同' 都没有找到相似词汇。
所有术语表中，词汇 '外长级' 都没有找到相似词汇。
所有术语表中，词汇 '对话' 都没有找到相似词汇。
所有术语表中，词汇 '会见' 都没有找到相似词汇。
所有术语表中，词汇 '第四次' 都没有找到相似词汇。
所有术语表中，词汇 '全面' 都没有找到相似词汇。
所有术语表中，词汇 '外长' 的相似词汇：
在术语表 '/content/1.22_zh-CN_en-US107行.xlsx' 中，相似词汇是：
{'zh': '外交部长', 'en': 'Foreign Minister'}，相似度：0.6624780073030224
所有术语表中，词汇 '举行' 都没有找到相似词汇。
所有术语表中，词汇 '维埃拉' 的相似词汇：
在术语表 '/content/1.22_zh-CN_en-US107行.xlsx' 中，相似词汇是：
{'zh': '维埃拉', 'en': 'Mauro Vieira'}，相似度：0.9999998638455286
所有术语表中，词汇 '战略' 的相似词汇：
在术语表 '/c

In [16]:
appid = '20240508002045108'
appkey = 'nEtBFE6G38kp40WoEE49'

from_lang = 'auto'
to_lang = 'auto'

endpoint = 'http://api.fanyi.baidu.com'
path = '/api/trans/vip/translate'
url = endpoint + path

query = f"{input_sentence}"

salt = random.randint(32768, 65536)
sign = md5((appid + query + str(salt) + appkey).encode()).hexdigest()

headers = {'Content-Type': 'application/x-www-form-urlencoded'}
payload = {'appid': appid, 'q': query, 'from': from_lang, 'to': to_lang, 'salt': salt, 'sign': sign}

r = requests.post(url, data=payload, headers=headers)
result = r.json()

Baidutranslation = result['trans_result'][0]['dst']

time.sleep(1)  # 等待1秒后再发送下一个请求

In [17]:
# -*- coding: utf-8 -*-
import sys
import uuid
import hashlib
from imp import reload

reload(sys)

YOUDAO_URL = 'https://openapi.youdao.com/api'
APP_KEY = '600d7a10ac6321e6'
APP_SECRET = 'ABlMoLgW7YZNZfgFq1sbL4fuOHZde8Bw'


def encrypt(signStr):
    hash_algorithm = hashlib.sha256()
    hash_algorithm.update(signStr.encode('utf-8'))
    return hash_algorithm.hexdigest()


def truncate(q):
    if q is None:
        return None
    size = len(q)
    return q if size <= 20 else q[0:10] + str(size) + q[size - 10:size]


def do_request(data):
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    return requests.post(YOUDAO_URL, data=data, headers=headers)


def connect():
    q = f"{input_sentence}"

    data = {}
    data['from'] = 'auto'
    data['to'] = 'auto'
    data['signType'] = 'v3'
    curtime = str(int(time.time()))
    data['curtime'] = curtime
    salt = str(uuid.uuid1())
    signStr = APP_KEY + truncate(q) + salt + curtime + APP_SECRET
    sign = encrypt(signStr)
    data['appKey'] = APP_KEY
    data['q'] = q
    data['salt'] = salt
    data['sign'] = sign
    data['vocabId'] = "您的用户词表ID"

    response = do_request(data)
    contentType = response.headers['Content-Type']
    if contentType == "audio/mp3":
        millis = int(round(time.time() * 1000))
        filePath = "合成的音频存储路径" + str(millis) + ".mp3"
        fo = open(filePath, 'wb')
        fo.write(response.content)
        fo.close()
    else:
        print(response.content)
        print(response.text)
    return response.text

string=connect()

# 将JSON字符串解析为Python字典

Youdaotranslation = json.loads(string).get("translation", [""])[0]


b'{"tSpeakUrl":"https://openapi.youdao.com/ttsapi?q=They+held+the+fourth+China-Brazil+Comprehensive+Strategic+Dialogue+at+the+foreign+ministers%27+level+and+jointly+met+the+press+with+Brazilian+Foreign+Minister+Joao+Vieira.&langType=en-USA&sign=66A90C07EA36906D687689595178C176&salt=1718763373321&voice=4&format=mp3&appKey=600d7a10ac6321e6&ttsVoiceStrict=false&osType=api","requestId":"5b78995d-3956-454d-831b-d5dd1f257af9","query":"\xe5\x90\x8c\xe5\xb7\xb4\xe8\xa5\xbf\xe5\xa4\x96\xe9\x95\xbf\xe7\xbb\xb4\xe5\x9f\x83\xe6\x8b\x89\xe4\xb8\xbe\xe8\xa1\x8c\xe4\xba\x86\xe7\xac\xac\xe5\x9b\x9b\xe6\xac\xa1\xe4\xb8\xad\xe5\xb7\xb4\xe5\xa4\x96\xe9\x95\xbf\xe7\xba\xa7\xe5\x85\xa8\xe9\x9d\xa2\xe6\x88\x98\xe7\x95\xa5\xe5\xaf\xb9\xe8\xaf\x9d\xe5\xb9\xb6\xe5\x85\xb1\xe5\x90\x8c\xe4\xbc\x9a\xe8\xa7\x81\xe8\xae\xb0\xe8\x80\x85\xe3\x80\x82","translation":["They held the fourth China-Brazil Comprehensive Strategic Dialogue at the foreign ministers\' level and jointly met the press with Brazilian Foreign Mini

In [18]:
!huggingface-cli login

# 下载并加载翻译质量评估模型
model_path = download_model("Unbabel/wmt22-cometkiwi-da")
model = load_from_checkpoint(model_path)


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: read).
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in c

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [19]:
data_Baidu = {"src": f"{input_sentence}", "mt": f"{Baidutranslation}"}
model_output_mt_Baidu = model.predict([data_Baidu], batch_size=8)
print(model_output_mt_Baidu)

data_Youdao = {"src": f"{input_sentence}", "mt": f"{Youdaotranslation}"}
model_output_mt_Youdao = model.predict([data_Youdao], batch_size=8)
print(model_output_mt_Youdao)

def extract_scores(data_str):
    pattern = r"'scores', \[([0-9.]+)\]"
    match = re.search(pattern, data_str)
    if match:
        # 提取匹配的内容并转换为浮点数
        scores_value = float(match.group(1))
        return scores_value
    return None

Score_Baidu = extract_scores(f"{model_output_mt_Baidu}")
Score_Youdao = extract_scores(f"{model_output_mt_Youdao}")

if Score_Baidu > Score_Youdao:
  Better_translation=Baidutranslation
else:
  Better_translation=Youdaotranslation
print('\n')
print(f"百度翻译：{Baidutranslation}")
print(f"得分：{Score_Baidu}")
print(f"有道翻译：{Youdaotranslation}")
print(f"得分：{Score_Youdao}")

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Prediction([('scores', [0.8012422323226929]), ('system_score', 0.8012422323226929)])


Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]


Prediction([('scores', [0.8082499504089355]), ('system_score', 0.8082499504089355)])


百度翻译：We held the fourth comprehensive strategic dialogue at the ministerial level with Brazilian Foreign Minister Vieira and jointly met with reporters.
得分：0.8012422323226929
有道翻译：They held the fourth China-Brazil Comprehensive Strategic Dialogue at the foreign ministers' level and jointly met the press with Brazilian Foreign Minister Joao Vieira.
得分：0.8082499504089355


In [20]:
# 构建新的Prompt
prompt = (
    f"请基于以下翻译记忆【中文：{best_translation_memory['zh']}；英文：{best_translation_memory['en']}】，"
    f"与以下机器翻译【{Better_translation}】，"
    f"以及以下术语【{terms_json}】，"
    f"来翻译下面的句子：{input_sentence}"
    f'请你严格按照格式输出翻译内容，参考样式为："Translation:（翻译好的文本）"，'
    f'注意：给出的句子若为英文，则翻译成中文；若为中文，则翻译成英文'

)
# 打印Prompt
print(prompt)

# 尝试问一个翻译的问题
response = client.chat.completions.create(
    model="glm-4",  # 填写需要调用的模型名称
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message)

请基于以下翻译记忆【中文：同巴西外长维埃拉举行了第四次中巴外长级全面战略对话并共同会见记者。；英文：He also co-chaired the fourth China-Brazil Foreign Ministerial-Level Comprehensive Strategic Dialogue with Foreign Minister Mauro Vieira and jointly met the press.】，与以下机器翻译【They held the fourth China-Brazil Comprehensive Strategic Dialogue at the foreign ministers' level and jointly met the press with Brazilian Foreign Minister Joao Vieira.】，以及以下术语【[
    {
        "zh": "无国界记者",
        "en": "Reporters Without Borders"
    },
    {
        "zh": "新华社",
        "en": "Xinhua News Agency"
    },
    {
        "zh": "例行记者会",
        "en": "Regular Press Conference"
    },
    {
        "zh": "外交部长",
        "en": "Foreign Minister"
    },
    {
        "zh": "维埃拉",
        "en": "Mauro Vieira"
    },
    {
        "zh": "战略伙伴关系",
        "en": "strategic partnership"
    },
    {
        "zh": "中巴建交50周年",
        "en": "50th anniversary of diplomatic relations between China and Brazil"
    },
    {
        "zh": "巴西",
        "en": "Brazi

In [21]:
# 正则表达式模式，匹配 "Translation" 后的文本内容
pattern = r'Translation:\s*(.*?)\s*(?:\n\n|$)'

# 搜索匹配的内容
match = re.search(pattern, response.choices[0].message.content)

# 提取匹配的文本内容
if match:
    Best_translation = match.group(1)
else:
    print("No match found")

data_Besttranslation = {"src": f"{input_sentence}", "mt": f"{Best_translation}"}
model_output_mt_Best = model.predict([data_Besttranslation], batch_size=8)

print("\n")
print(f"AI翻译：{Best_translation}")
print("最终评分为：",model_output_mt_Best)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.06s/it]



AI翻译：He held the fourth China-Brazil Comprehensive Strategic Dialogue at the foreign ministers' level with Brazilian Foreign Minister Mauro Vieira and jointly met the press.
最终评分为： Prediction([('scores', [0.7907233834266663]), ('system_score', 0.7907233834266663)])
